# Chapter 9 (companion): Alignment & Advanced Training Strategies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kimfalk/modern-recommender-systems/blob/main/notebooks/chapter-09/gen_recsys_alignment.ipynb)

This notebook covers the advanced techniques from the chapter, building on the fine-tuned model from `gen_recsys.ipynb`:

1. **Contrastive learning** — softening cross-entropy's harsh treatment of near misses (Listing 9.15)
2. **Direct Preference Optimization (DPO)** — aligning with user preferences via a frozen reference model (Listings 9.21–9.22)
3. **In-context learning (ICL)** — recommendations and reranking without fine-tuning (Listings 9.23–9.24)
4. **Anchored prompts** — bridging world knowledge and semantic IDs (Listing 9.25)
5. **Sampled softmax** — efficient decoding at scale (Listing 9.26)

> **Prerequisite:** run `gen_recsys.ipynb` first, or run its setup cells here. This notebook assumes `model`, `tokenizer`, `formatter`, `item_df`, and `train_data` exist. The setup cell below restores them from the SFT checkpoint.

In [ ]:
# Environment Setup
from recsys.utils.colab import setup_colab_environment, get_data_path, check_gpu

setup_colab_environment()
check_gpu()

In [ ]:
# Imports and state restore
import copy
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Restore the SFT checkpoint produced by gen_recsys.ipynb.
# Adjust the path if you changed output_dir in the training arguments.
SFT_CHECKPOINT = "./recsys_gpt"

try:
    tokenizer = GPT2Tokenizer.from_pretrained(SFT_CHECKPOINT)
    model = GPT2LMHeadModel.from_pretrained(SFT_CHECKPOINT)
    print("Loaded fine-tuned checkpoint.")
except OSError:
    print("SFT checkpoint not found - run gen_recsys.ipynb first, "
          "or point SFT_CHECKPOINT at a saved checkpoint directory.")

In [ ]:
# The BaseFormatter from gen_recsys.ipynb (repeated here so this
# notebook can run standalone once item_df is loaded)
class BaseFormatter:
    def __init__(self, item_df):
        self.item_map = dict(zip(item_df['id'], item_df['final_id']))
        self.tokens_missing = Counter()

    def get_item_tokens(self, item_uuid):
        if item_uuid not in self.item_map:
            return []
        s = self.item_map[item_uuid]
        return [f"L1_{s[0]}", f"L2_{s[1]}", f"L3_{s[2]}", f"LF_{s[3]}"]

    def format(self, user_record, is_training=True):
        tokens = []
        for t in user_record['history']:
            if t in self.item_map:
                tokens.extend(self.get_item_tokens(t))
            else:
                self.tokens_missing.update([t])
        return " ".join(tokens)

# Load item_df / train_data the same way as in gen_recsys.ipynb
# (real artifacts or the synthetic fallback). Placeholder here:
# item_df = pd.read_parquet(...); train_data = pd.read_parquet(...)
# formatter = BaseFormatter(item_df)

## 9.3.2 Improving Semantic IDs with Contrastive Learning

Cross-entropy is the default loss for causal LMs — it is built into the model's `forward()` pass, no configuration needed. But it punishes near misses as harshly as complete misses. Contrastive learning adds a second objective over three vectors:

- **Anchor** — hidden state of the last token in the user's history
- **Positive** — hidden state of the item the user actually interacted with next
- **Negative** — hidden state of an item they did not

The triplet loss pulls anchor and positive together and pushes the negative away. To use it, replace the built-in `Trainer` loss with a custom training loop that calls `compute_combined_loss`. For **hard negatives**, sample items whose semantic IDs share a prefix with the positive but differ at the leaf — same neighborhood, genuinely challenging comparison.

In [ ]:
# Listing 9.15: Combined generative and contrastive loss
contrastive_loss_fn = nn.TripletMarginLoss(margin=1.0, p=2)  #A

def compute_combined_loss(model,
                          user_history_ids,
                          true_next_item_id,
                          negative_item_id):
    outputs_hist = model(user_history_ids, output_hidden_states=True)
    anchor_embed = outputs_hist.hidden_states[-1][:, -1, :]  #B

    outputs_pos = model(true_next_item_id, output_hidden_states=True)
    pos_embed = outputs_pos.hidden_states[-1][:, -1, :]  #C

    outputs_neg = model(negative_item_id, output_hidden_states=True)
    neg_embed = outputs_neg.hidden_states[-1][:, -1, :]  #D

    loss_cl = contrastive_loss_fn(anchor_embed, pos_embed, neg_embed)  #E

    logits = model(user_history_ids).logits
    loss_gen = F.cross_entropy(
        logits[:, -1, :], true_next_item_id.view(-1)
    )  #F

    lambda_cl = 0.1  #G
    total_loss = loss_gen + (lambda_cl * loss_cl)  #H
    return total_loss

#A Triplet margin loss function
#B Anchor: last hidden state of user history
#C Positive: hidden state of true next item
#D Negative: hidden state of random item
#E Contrastive loss
#F Standard cross-entropy loss
#G Contrastive weight
#H Combined loss

In [ ]:
# Hard negative mining: same semantic prefix, different leaf
def mine_hard_negative(item_df, positive_uuid, formatter):
    pos_id = formatter.item_map.get(positive_uuid)
    if pos_id is None:
        return None
    prefix = pos_id[:3]  # (L1, L2, L3)
    candidates = item_df[
        item_df['final_id'].apply(lambda t: t[:3] == prefix)
        & (item_df['id'] != positive_uuid)
    ]
    if len(candidates) == 0:
        # Fall back to sharing only (L1, L2), then random
        candidates = item_df[
            item_df['final_id'].apply(lambda t: t[:2] == prefix[:2])
            & (item_df['id'] != positive_uuid)
        ]
    if len(candidates) == 0:
        return item_df.sample(1)['id'].iloc[0]
    return candidates.sample(1)['id'].iloc[0]

## 9.6 Aligning with User Preferences via DPO

Predicting what a user will click is not the same as predicting what they prefer. DPO aligns the model directly on preference triplets — *(prompt, chosen, rejected)* — constructed from interaction logs: clicked items become *chosen*, ignored or disliked items become *rejected*.

The frozen **reference model** anchors training. Without it, the policy model could reward-hack — e.g., emit `L1_5 L1_5 L1_5` because repeating popular tokens maximizes likelihood while breaking the semantic ID grammar. `beta` (0.1 is typical) controls how far the policy may drift from the reference.

In [ ]:
# Listing 9.21: DPO loss function
class DPOLoss(nn.Module):
    def __init__(self, beta=0.1):
        super().__init__()
        self.beta = beta  #A

    def forward(self,
                policy_chosen_logps,
                policy_rejected_logps,
                ref_chosen_logps,
                ref_rejected_logps):  #B
        pi_logratios = (policy_chosen_logps
                        - policy_rejected_logps)  #C
        ref_logratios = (ref_chosen_logps
                         - ref_rejected_logps)
        logits = pi_logratios - ref_logratios  #D
        losses = -F.logsigmoid(self.beta * logits)
        return losses.mean()

#A Controls how far the model can deviate from the base
#B Requires log-probabilities from both models
#C Gap between chosen and rejected in the policy model
#D Compare against the same gap in the reference model

In [ ]:
# Initialization: load the SFT checkpoint twice -
# once trainable (policy), once frozen (reference)
policy_model = model
ref_model = copy.deepcopy(model)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

dpo_loss = DPOLoss(beta=0.1)
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=1e-5)

In [ ]:
# Helper: sum of log-probabilities of candidate tokens given a prompt.
# This is the concrete implementation of the get_logprobs used in
# the pseudocode of Listing 9.22 - the same slicing idea as the
# perplexity evaluator's calculate_score.
def get_logprobs(model, prompt, candidate, tokenizer, device=None):
    device = device or model.device
    full_text = prompt + " " + candidate
    input_ids = tokenizer(full_text, return_tensors="pt").input_ids.to(device)
    prompt_len = len(tokenizer.encode(prompt, add_special_tokens=False))

    outputs = model(input_ids)
    shift_logits = outputs.logits[..., :-1, :]
    shift_labels = input_ids[..., 1:]

    cand_logits = shift_logits[:, prompt_len:, :]
    cand_labels = shift_labels[:, prompt_len:]

    logps = F.log_softmax(cand_logits, dim=-1)
    token_logps = torch.gather(
        logps, 2, cand_labels.unsqueeze(-1)
    ).squeeze(-1)
    return token_logps.sum(dim=-1)

In [ ]:
# Listing 9.22: DPO training step
def train_dpo_step(batch):
    with torch.no_grad():  #A
        ref_chosen_logps = get_logprobs(
            ref_model, batch['prompt'], batch['chosen'], tokenizer)
        ref_rejected_logps = get_logprobs(
            ref_model, batch['prompt'], batch['rejected'], tokenizer)

    policy_chosen_logps = get_logprobs(
        policy_model, batch['prompt'], batch['chosen'], tokenizer)  #B
    policy_rejected_logps = get_logprobs(
        policy_model, batch['prompt'], batch['rejected'], tokenizer)

    loss = dpo_loss(
        policy_chosen_logps, policy_rejected_logps,
        ref_chosen_logps, ref_rejected_logps)  #C

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

#A Frozen reference model, no gradients
#B Active policy model, gradients enabled
#C DPO loss compares preference gaps

In [ ]:
# Constructing preference triplets from feedback data and running DPO.
# In production, chosen = clicked / liked, rejected = ignored / disliked
# from your interaction logs. Here we build triplets from user histories:
# the true next item is 'chosen', a random unseen item is 'rejected'.
def build_preference_triplets(train_data, formatter, item_df, n=200):
    triplets = []
    all_items = list(item_df['id'])
    for _, user in train_data.iterrows():
        history = user['history']
        if len(history) < 3:
            continue
        ctx_user = user.copy()
        ctx_user['history'] = history[:-1]
        prompt = formatter.format(ctx_user, is_training=False)
        chosen = " ".join(formatter.get_item_tokens(history[-1]))
        rejected_uuid = random.choice(all_items)
        while rejected_uuid in set(history):
            rejected_uuid = random.choice(all_items)
        rejected = " ".join(formatter.get_item_tokens(rejected_uuid))
        if chosen and rejected:
            triplets.append(
                {"prompt": prompt, "chosen": chosen, "rejected": rejected})
        if len(triplets) >= n:
            break
    return triplets

# triplets = build_preference_triplets(train_data, formatter, item_df)
# for step, batch in enumerate(triplets):
#     loss = train_dpo_step(batch)
#     if step % 50 == 0:
#         print(f"step {step}: DPO loss {loss:.4f}")

## 9.7 In-Context Learning: Recommendations Without Fine-Tuning

ICL adapts a model by showing it examples in the prompt — no gradient updates, no semantic IDs, no infrastructure. It leverages the LLM's world knowledge directly, which makes it the natural complement to fine-tuning: strong on cold-start and short-term interest, weaker on catalog alignment.

In [ ]:
# Listing 9.23: ICL prompt for movie recommendations
icl_prompt = """
You are a movie recommender. Given a user's watch history,
recommend the next movie they should watch.

Example 1:
History: Interstellar, Arrival, Blade Runner 2049
Next: Dune

Example 2:
History: The Dark Knight, Se7en, Prisoners
Next: Zodiac

Now recommend for this user:
History: Annihilation, Ex Machina, Coherence
Next:"""

print(icl_prompt)

### 9.7.1 ICL as a Reranking Layer

The most effective production pattern: your fine-tuned (or traditional) retrieval model generates catalog-safe candidates, and a large instruction-tuned model reranks them with recent history and context in the prompt. Your fine-tuned DistilGPT2 would not work as the reranker — it has learned semantic IDs at the cost of much of its natural-language reasoning.

Watch out for **position bias**: LLMs favor items that appear earlier in a list. Randomize candidate order across requests, or use pointwise scoring (one candidate per call) to eliminate it entirely.

In [ ]:
# Listing 9.24: ICL reranking over a candidate set
def icl_rerank(candidates, user_history,
               user_context, llm):
    history_str = ", ".join(user_history[-5:])  #A
    candidates_str = "\n".join(
        [f"{i+1}. {c}" for i, c in enumerate(candidates)]
    )  #B

    prompt = f"""You are a personalized recommender.
User's recent watch history: {history_str}
Context: {user_context}

From the following candidates, rank the top 3
most relevant:
{candidates_str}

Top 3 recommendations:"""

    return llm.generate(prompt)  #C

#A Use the last 5 items for recency
#B Format candidates as a numbered list
#C Call the LLM API (e.g., GPT-4, Claude)

## 9.8 Bridging World Knowledge and Semantic IDs

Replacing titles with opaque tokens severs the model's world knowledge from the tokens it must predict — the **semantic gap**. Anchored prompts narrow it by pairing each semantic ID with a short natural-language descriptor from the item's metadata, so the model repeatedly sees concept and ID together and learns the association.

Limitations to keep in mind: anchors are only as good as your metadata; the model still cannot reason flexibly about nuanced preferences; and new items remain uncertain. The full solution — separating language reasoning from catalog grounding — is the next chapter's RAG architecture.

In [ ]:
# Listing 9.25: Pairing descriptions with semantic IDs
class AnchoredFormatter(BaseFormatter):
    def __init__(self, item_df):
        super().__init__(item_df)
        self.item_descriptions = dict(
            zip(item_df['id'], item_df['genre_tags'])  #A
        )

    def format(self, user_record, is_training=True):
        tokens = []

        preferences = user_record.get('stated_preferences', [])
        for pref in preferences:
            tokens.append(f"[LIKES_{pref.upper()}]")  #B

        for item_uuid in user_record['history']:
            description = self.item_descriptions.get(
                item_uuid, "UNKNOWN"
            )
            tokens.append(f"[{description}]")  #C
            tokens.extend(self.get_item_tokens(item_uuid))  #D

        return " ".join(tokens)

#A Map item UUIDs to short genre descriptors
#B Stated preferences become tokens like [LIKES_SCIFI]
#C Natural language anchor before each item
#D Semantic IDs follow immediately

## 9.9.2 Efficient Decoding at Scale

The final softmax computes `d × V` operations over the whole vocabulary. Sampled softmax computes logits only for the targets plus a random sample of negatives — typically ~10K instead of the full vocabulary, up to 100x faster during training. This listing is illustrative; integrating it with the model's forward pass requires careful implementation.

In [ ]:
# Listing 9.26: Sampled softmax loss (illustrative)
class SampledSoftmaxLoss(nn.Module):
    def __init__(self, embedding, num_samples=10000):
        super().__init__()
        self.embedding = embedding  #A
        self.num_samples = num_samples

    def forward(self, hidden_states, target_ids,
                full_vocab_size):
        sampled_ids = target_ids.clone()  #B
        num_negatives = self.num_samples - len(target_ids)
        negatives = torch.randint(
            0, full_vocab_size, (num_negatives,)
        )
        sampled_ids = torch.cat([sampled_ids, negatives])  #C

        sampled_weights = self.embedding.weight[sampled_ids]
        logits = torch.matmul(
            hidden_states, sampled_weights.T
        )

        target_positions = torch.arange(len(target_ids))
        return F.cross_entropy(logits, target_positions)  #D

#A Reference to the model's output embedding layer
#B Always include target items
#C Add random negative samples
#D Cross-entropy over the reduced vocabulary

## Summary

- Contrastive learning teaches the model that semantically similar items are acceptable alternatives; hard negatives (same prefix, different leaf) make the signal sharper.
- DPO widens the preference gap between chosen and rejected items, with a frozen reference model preventing reward hacking that would break semantic ID formatting.
- ICL is a zero-infrastructure alternative, best deployed as a reranking layer over catalog-safe candidates from a fine-tuned or traditional retrieval model.
- Anchored prompts partially bridge the semantic gap; the RAG architecture in the next chapter dissolves it.